[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

# Protocol Buffers Primer — Deep Dive

| # | Section | Description |
|---|---------|-------------|
| 1 | [What are Protocol Buffers?](#1-what-are-protocol-buffers) | History, design goals, schema-first philosophy |
| 2 | [Why ONNX Uses Protobuf](#2-why-onnx-uses-protobuf) | Performance, size, cross-language, evolution |
| 3 | [Proto File Syntax](#3-proto-file-syntax) | Messages, fields, types, enums, nesting |
| 4 | [Wire Format and Encoding](#4-wire-format-and-encoding) | Varint math, wire types, tag encoding |
| 5 | [Varint Encoding — Worked Example](#5-varint-encoding--worked-example) | Step-by-step numerical walkthrough |
| 6 | [ZigZag Encoding for Signed Integers](#6-zigzag-encoding-for-signed-integers) | Mapping signed to unsigned efficiently |
| 7 | [Tag Encoding and Field Numbering](#7-tag-encoding-and-field-numbering) | Tag computation and best practices |
| 8 | [Serialization Round-Trip](#8-serialization-round-trip) | Serialize, inspect bytes, deserialize |
| 9 | [Size Comparison: Protobuf vs JSON vs XML](#9-size-comparison) | Empirical benchmarks |
| 10 | [Visualizing Protobuf Message Trees](#10-visualizing-protobuf-message-trees) | ONNX message hierarchy diagram |
| 11 | [Schema Evolution and Compatibility](#11-schema-evolution-and-compatibility) | Forward/backward compat rules |
| 12 | [Serialization Complexity Analysis](#12-serialization-complexity-analysis) | Big-O analysis of encode/decode |
| 13 | [Key Takeaways](#13-key-takeaways) | Summary of core concepts |

In [ ]:
# Install dependencies (uncomment in Colab)
# !pip install onnx numpy matplotlib networkx --quiet

## 1. What are Protocol Buffers?

**Protocol Buffers** (Protobuf) are a language-neutral, platform-neutral, extensible mechanism for serializing structured data, developed at Google and open-sourced in 2008. The core idea is **schema-first design**: you describe your data structures in a `.proto` definition file, then a compiler (`protoc`) generates efficient serialization/deserialization code for your target language — C++, Python, Java, Go, Rust, and many others.

Unlike text-based formats such as JSON or XML, Protobuf produces a **compact binary representation** where field names are replaced by small integer tags and values are stored in their native binary form. This yields dramatically smaller payloads and faster parsing, which is critical for ML model files that can contain millions of weight parameters.

### The Protobuf Lifecycle

The Protobuf lifecycle follows a **define → compile → serialize** pipeline:

```
┌─────────────────┐      ┌──────────────┐      ┌───────────────────┐      ┌──────────────────┐
│  .proto file     │─────▶│   protoc      │─────▶│  Generated        │─────▶│  Binary bytes     │
│  (schema)        │      │   compiler    │      │  Python/C++/Java  │      │  (wire format)    │
│                  │      │               │      │  classes          │      │                   │
└─────────────────┘      └──────────────┘      └───────────────────┘      └──────────────────┘
         │                                              │                           │
         │            Human-readable                    │       Type-safe API       │     Compact
         │            schema definition                 │       for your language    │     on-disk / wire
```

In the ONNX ecosystem, the `.proto` file ships pre-compiled — the `onnx` Python package includes generated Python classes (`ModelProto`, `GraphProto`, `NodeProto`, etc.) so users never need to run `protoc` manually. When you call `onnx.load("model.onnx")`, you are invoking Protobuf's `ParseFromString` under the hood, reconstructing a full object graph from compact binary bytes.

### Proto2 vs Proto3

Protobuf supports two major syntax versions:

| Feature | proto2 | proto3 |
|---------|--------|--------|
| Field presence | Explicit `required`/`optional` | All fields optional by default |
| Default values | User-defined defaults | Language-specific zero values |
| Unknown fields | Preserved on parse | Preserved (since 3.5+) |
| Enum zero value | Not required | Must have zero value |
| Maps | Available | Available |

ONNX uses **proto3 syntax** with some proto2-compatible patterns, which is important to understand when reading the ONNX specification.

## 2. Why ONNX Uses Protobuf

Several design forces push ONNX toward a binary, schema-first format rather than JSON, XML, or a custom binary format:

### 2.1 Performance

Parsing binary Protobuf is $O(n)$ in the size of the serialized data with minimal branching — there are no string delimiters to scan for, no escape sequences to decode, and no whitespace to skip. Benchmarks consistently show Protobuf parsing is 20–100× faster than JSON for structured data, which matters when loading multi-gigabyte transformer models.

The parsing cost can be modeled as:

$$T_{\text{parse}} = \sum_{i=1}^{N_{\text{fields}}} \bigl(c_{\text{tag}} + c_{\text{decode}}(\text{type}_i, \text{size}_i)\bigr)$$

where $c_{\text{tag}}$ is the constant cost of reading a tag varint, and $c_{\text{decode}}$ depends on the wire type (varint decoding, fixed-width copy, or length-prefixed read).

### 2.2 Compactness

Protobuf replaces human-readable field names with small integer *field tags* encoded as 1–2 byte varints. Repeated numeric fields use *packed encoding*, storing values back-to-back without per-element overhead. For a model with $N$ float32 weights:

$$\text{Protobuf size} \approx 4N + O(\text{metadata})$$
$$\text{JSON size} \approx 10N + O(\text{keys})$$

For 100 million float32 weights, the overhead beyond the raw 400 MB of weight data is typically less than 1%.

### 2.3 Cross-language Tooling

Protobuf's code generation is mature across C++ (used by ONNX Runtime's core), Python (used for model authoring), Java/Kotlin (mobile deployment), Go, Rust, and C#. Every language gets type-safe accessors generated from the same `.proto` source, eliminating hand-written parsers and the bugs they introduce.

### 2.4 Schema Evolution

Protobuf's compatibility story — never reuse a field number, never change a field type, new fields get default values — maps well to ONNX's versioning needs. A runtime compiled against OpSet 13 can still *parse* a model that includes OpSet 17 nodes (it may not *execute* them, but it won't crash on parsing).

### 2.5 Spec Clarity

The `.proto` file is an unambiguous, machine-readable contract. There is exactly one source of truth for what fields `ModelProto` contains, what types they are, and what tag numbers identify them on the wire.

## 3. Proto File Syntax

A `.proto` file defines **messages** (struct-like records), **enums** (named integer constants), and **services** (RPC endpoints, not used by ONNX). Below is a simplified example illustrating the key ingredients:

```protobuf
syntax = "proto3";
package tutorial.example;

message TensorDesc {
  string name       = 1;   // field tag 1, wire type 2 (length-delimited)
  repeated int64 shape = 2; // field tag 2, packed varint array
  enum DType {
    DTYPE_UNKNOWN = 0;     // proto3 requires 0 as default
    FLOAT         = 1;
    INT64         = 2;
  }
  DType data_type    = 3;   // field tag 3, wire type 0 (varint)
}
```

### Field Rules

| Keyword | Meaning | Wire behavior |
|---------|---------|---------------|
| *(default)* | Singular, implicit presence | Omitted when zero-value |
| `optional` | Explicit presence tracking | `has_field()` returns `True`/`False` |
| `repeated` | Ordered list (0 or more) | Packed encoding for scalars |
| `map<K,V>` | Associative container | Repeated key-value pair messages |
| `oneof` | Mutually exclusive fields | At most one field set |

### Scalar Types and Their Wire Representations

| Proto type | Python type | Size | Wire type |
|-----------|-------------|------|-----------|
| `int32`, `int64` | `int` | Variable (varint) | 0 |
| `uint32`, `uint64` | `int` | Variable (varint) | 0 |
| `sint32`, `sint64` | `int` | Variable (ZigZag + varint) | 0 |
| `float` | `float` | 4 bytes | 5 |
| `double` | `float` | 8 bytes | 1 |
| `bool` | `bool` | 1 byte varint | 0 |
| `string` | `str` | Length-prefixed UTF-8 | 2 |
| `bytes` | `bytes` | Length-prefixed raw | 2 |

### Field Numbering Best Practices

**Field numbers** (`= 1`, `= 2`, ...) are the wire identifiers. They must be unique within a message and should **never be reused** once assigned, even if the field is deleted. Numbers 1–15 encode in a single byte (tag), making them ideal for frequently-set fields.

```
Field number encoding cost:
┌──────────────────┬────────────────┬──────────────────────────┐
│  Field numbers   │  Tag bytes     │  Recommendation          │
├──────────────────┼────────────────┼──────────────────────────┤
│  1 – 15          │  1 byte        │  Hot / frequent fields   │
│  16 – 2047       │  2 bytes       │  Normal fields           │
│  2048 – 262143   │  3 bytes       │  Rare / extension fields │
└──────────────────┴────────────────┴──────────────────────────┘
```

In [ ]:
import onnx
import os

proto_path = os.path.join(os.path.dirname(onnx.__file__), 'onnx.proto')
print(f"onnx.proto location: {proto_path}")
print(f"File exists: {os.path.exists(proto_path)}")

if os.path.exists(proto_path):
    with open(proto_path) as f:
        content = f.read()
    messages = [line.strip() for line in content.split('\n') if line.strip().startswith('message ')]
    enums = [line.strip() for line in content.split('\n') if line.strip().startswith('enum ')]
    print(f"\nMessage definitions ({len(messages)}):")
    for m in messages:
        print(f"  {m}")
    print(f"\nEnum definitions ({len(enums)}):")
    for e in enums:
        print(f"  {e}")

## 4. Wire Format and Encoding

When Protobuf serializes a message, each field is stored as a **(tag, value)** pair. The tag encodes both the **field number** and the **wire type** in a single varint:

$$\text{tag} = (\text{field\_number} \ll 3) \;|\; \text{wire\_type}$$

This means the low 3 bits identify *how* to read the value (its wire type), while the remaining bits identify *which* field it belongs to.

### Wire Types

```
┌────────────┬───────┬────────────────────────────────────────────────────────────────┐
│ Wire Type  │ Value │ Used For                                                       │
├────────────┼───────┼────────────────────────────────────────────────────────────────┤
│ Varint     │   0   │ int32, int64, uint32, uint64, sint32, sint64, bool, enum       │
│ 64-bit     │   1   │ fixed64, sfixed64, double                                      │
│ Len-delim  │   2   │ string, bytes, embedded messages, packed repeated               │
│ 32-bit     │   5   │ fixed32, sfixed32, float                                       │
└────────────┴───────┴────────────────────────────────────────────────────────────────┘
```

### Varint Encoding — The Core Primitive

Varints use a **continuation bit** scheme: each byte contributes 7 payload bits, and the MSB indicates whether more bytes follow:

$$\text{value} = \sum_{i=0}^{n-1} (b_i \;\&\; \texttt{0x7F}) \cdot 128^i$$

Equivalently, using bit shifts:

$$\text{value} = \sum_{i=0}^{n-1} (b_i \;\&\; \texttt{0x7F}) \ll (7i)$$

where $b_i$ is the $i$-th byte and $n$ is the number of bytes consumed (determined by finding the first byte with MSB = 0).

**Properties of varint encoding:**

| Value range | Bytes needed | Bit budget |
|------------|-------------|------------|
| $0$ – $127$ ($2^7 - 1$) | 1 | 7 bits |
| $128$ – $16{,}383$ ($2^{14} - 1$) | 2 | 14 bits |
| $16{,}384$ – $2{,}097{,}151$ ($2^{21} - 1$) | 3 | 21 bits |
| $2^{21}$ – $2^{28} - 1$ | 4 | 28 bits |
| ... | ... | ... |
| $2^{56}$ – $2^{63} - 1$ | 9 | 63 bits |
| $2^{63}$ – $2^{64} - 1$ | 10 | 64 bits |

The general formula for bytes needed to encode value $v > 0$:

$$n_{\text{bytes}}(v) = \left\lceil \frac{\lfloor \log_2(v) \rfloor + 1}{7} \right\rceil$$

For `uint64`, the maximum encoding is **10 bytes** since $\lceil 64/7 \rceil = 10$.

**Critical caveat:** Negative `int32`/`int64` values are encoded as large unsigned values (always 10 bytes!) — use `sint32`/`sint64` with ZigZag encoding for signed values.

## 5. Varint Encoding — Worked Example

Let's encode the integer **300** step by step.

**Step 1:** Convert 300 to binary:

$$300 = 256 + 32 + 8 + 4 = 100101100_2$$

**Step 2:** Split into 7-bit groups from LSB:

```
Original bits:   1  0010  1100
                 ─────────────
Group 1 (LSB):      010  1100   = 44₁₀
Group 0 (MSB):   0  0000  10    = 2₁₀
```

**Step 3:** Set continuation bit (MSB) on all groups except the last:

```
Byte 0: 1|0101100  = 0xAC = 172₁₀   (MSB=1 → more bytes follow)
Byte 1: 0|0000010  = 0x02 =   2₁₀   (MSB=0 → last byte)
```

**Verification:**

$$\text{value} = (\texttt{0xAC} \;\&\; \texttt{0x7F}) \cdot 128^0 + (\texttt{0x02} \;\&\; \texttt{0x7F}) \cdot 128^1$$

$$= 44 \cdot 1 + 2 \cdot 128 = 44 + 256 = 300 \; \checkmark$$

So the integer 300 is encoded as the two-byte sequence `[0xAC, 0x02]`.

### Visual Walkthrough

```
  Integer 300 (binary: 100101100)
         │
         ▼ Split into 7-bit groups (LSB first)
  ┌──────────┐  ┌──────────┐
  │ 0101100  │  │ 0000010  │
  │ (44)     │  │ (2)      │
  └──────────┘  └──────────┘
         │              │
         ▼              ▼ Add continuation bits
  ┌──────────┐  ┌──────────┐
  │1|0101100 │  │0|0000010 │
  │ 0xAC     │  │ 0x02     │
  │ more→    │  │ ←last    │
  └──────────┘  └──────────┘
         │              │
         ▼              ▼
  Wire bytes: [0xAC, 0x02]
```

In [ ]:
def encode_varint(value: int) -> bytes:
    """Encode a non-negative integer as a Protobuf varint."""
    if value == 0:
        return b'\x00'
    result = bytearray()
    while value > 0x7F:
        result.append((value & 0x7F) | 0x80)
        value >>= 7
    result.append(value & 0x7F)
    return bytes(result)

def decode_varint(data: bytes) -> int:
    """Decode a Protobuf varint from bytes."""
    value = 0
    for i, byte in enumerate(data):
        value |= (byte & 0x7F) << (7 * i)
        if not (byte & 0x80):
            break
    return value

encoded = encode_varint(300)
print(f"Encoding 300:")
print(f"  Bytes (decimal): {list(encoded)}")
print(f"  Bytes (hex):     {' '.join(f'0x{b:02X}' for b in encoded)}")
print(f"  Binary:          {' '.join(f'{b:08b}' for b in encoded)}")
print(f"  Decoded back:    {decode_varint(encoded)}")

print("\n--- Varint sizes for various values ---")
print(f"{'Value':>25s}  {'Bytes':>6s}  {'Hex':>20s}")
print("-" * 58)
test_values = [0, 1, 127, 128, 300, 16383, 16384, 2**20, 2**28, 2**35, 2**63 - 1]
for v in test_values:
    enc = encode_varint(v)
    print(f"  {v:>22,d}  {len(enc):>4d}    [{enc.hex()}]")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

values = np.array([2**i for i in range(64)])
varint_sizes = np.array([len(encode_varint(int(v))) for v in values])
fixed_sizes = np.full_like(varint_sizes, 8)

fig, ax = plt.subplots(figsize=(10, 5))
ax.step(range(64), varint_sizes, where='mid', label='Varint encoding', linewidth=2, color='#2196F3')
ax.axhline(y=8, color='#F44336', linestyle='--', linewidth=1.5, label='fixed64 (always 8 bytes)')
ax.axhline(y=4, color='#FF9800', linestyle='--', linewidth=1.5, label='fixed32 (always 4 bytes)')

ax.fill_between(range(64), varint_sizes, alpha=0.15, color='#2196F3')
ax.set_xlabel('Bit position of highest set bit ($\\lfloor\\log_2(v)\\rfloor$)', fontsize=12)
ax.set_ylabel('Encoded size (bytes)', fontsize=12)
ax.set_title('Protobuf Varint Size vs. Value Magnitude', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.set_xlim(0, 63)
ax.set_ylim(0, 11)
ax.grid(True, alpha=0.3)

for threshold in [7, 14, 21, 28, 35, 42, 49, 56, 63]:
    ax.axvline(x=threshold, color='gray', linestyle=':', alpha=0.3)

plt.tight_layout()
plt.show()
print("Varint is more compact than fixed64 for values < 2^56 (8 groups of 7 bits).")
print("For values >= 2^56, varint uses 9-10 bytes — worse than fixed64.")

## 6. ZigZag Encoding for Signed Integers

Standard varint encoding treats all values as unsigned. For negative numbers using `int32`/`int64`, two's complement produces a large unsigned value (MSB is set), requiring the maximum 5 or 10 bytes. **ZigZag encoding** solves this by interleaving positive and negative values:

$$\text{ZigZag}(n) = (n \ll 1) \oplus (n \gg 31) \quad \text{(for sint32)}$$

$$\text{ZigZag}(n) = (n \ll 1) \oplus (n \gg 63) \quad \text{(for sint64)}$$

The inverse (decoding) is:

$$\text{ZigZag}^{-1}(z) = (z \ggg 1) \oplus -(z \;\&\; 1)$$

where $\ggg$ is logical (unsigned) right shift.

### ZigZag Mapping Table

```
Signed  →  ZigZag (unsigned)  →  Varint bytes needed
──────────────────────────────────────────────────────
   0    →    0                 →  1 byte
  -1    →    1                 →  1 byte
   1    →    2                 →  1 byte
  -2    →    3                 →  1 byte
   2    →    4                 →  1 byte
  -3    →    5                 →  1 byte
  ...   →   ...               →  ...
  63    →  126                 →  1 byte
 -64    →  127                 →  1 byte
  64    →  128                 →  2 bytes
  ...   →   ...               →  ...
```

The key insight: values with small **absolute magnitude** (whether positive or negative) map to small unsigned values, achieving compact varint encoding. Without ZigZag, $-1$ as `int64` would be $2^{64}-1$, requiring 10 bytes!

In [ ]:
def zigzag_encode(n: int, bits: int = 32) -> int:
    """ZigZag encode a signed integer."""
    return (n << 1) ^ (n >> (bits - 1))

def zigzag_decode(z: int) -> int:
    """ZigZag decode to signed integer."""
    return (z >> 1) ^ -(z & 1)

print(f"{'Signed':>10s}  {'ZigZag':>10s}  {'Varint bytes':>12s}  {'vs int32':>10s}")
print("-" * 50)
test_signed = [0, -1, 1, -2, 2, -64, 64, -128, 128, -1000, 1000, -100000]
for n in test_signed:
    z = zigzag_encode(n)
    zz_bytes = len(encode_varint(z))
    raw_bytes = len(encode_varint(n & 0xFFFFFFFF if n < 0 else n))
    savings = 'same' if zz_bytes == raw_bytes else f'{raw_bytes - zz_bytes:+d}'
    print(f"{n:>10d}  {z:>10d}  {zz_bytes:>10d}    {savings:>8s}")
    assert zigzag_decode(z) == n, f"Round-trip failed for {n}"

## 7. Tag Encoding and Field Numbering

Every field on the wire is preceded by a **tag** that identifies both the field number and the wire type:

$$\text{tag} = (\text{field\_number} \ll 3) \;|\; \text{wire\_type}$$

Since tags are themselves encoded as varints, the field number determines the tag's byte cost:

```
Tag byte budget:

field_number  1-15:    tag value  8-127     → 1-byte varint
field_number 16-2047:  tag value 128-16383  → 2-byte varint
field_number 2048+:    tag value 16384+     → 3+ byte varint
```

### Worked Examples

| Field | Wire type | Tag formula | Tag value | Hex | Varint bytes |
|-------|-----------|-------------|-----------|-----|-------------|
| 1 (varint) | 0 | $(1 \ll 3) | 0$ | 8 | `0x08` | 1 |
| 1 (string) | 2 | $(1 \ll 3) | 2$ | 10 | `0x0A` | 1 |
| 2 (varint) | 0 | $(2 \ll 3) | 0$ | 16 | `0x10` | 1 |
| 15 (varint) | 0 | $(15 \ll 3) | 0$ | 120 | `0x78` | 1 |
| 16 (varint) | 0 | $(16 \ll 3) | 0$ | 128 | `0x80 0x01` | 2 |
| 100 (string) | 2 | $(100 \ll 3) | 2$ | 802 | `0xA2 0x06` | 2 |

This is why Protobuf best practice recommends reserving field numbers **1–15** for the most frequently used fields — they cost only one byte of tag overhead.

In [ ]:
def decode_tag(tag_value: int):
    """Split a Protobuf tag into field number and wire type."""
    wire_type = tag_value & 0x07
    field_number = tag_value >> 3
    wire_names = {0: 'Varint', 1: '64-bit', 2: 'Length-delimited', 5: '32-bit'}
    return field_number, wire_type, wire_names.get(wire_type, 'Unknown')

def make_tag(field_number: int, wire_type: int) -> int:
    return (field_number << 3) | wire_type

print("Tag encoding examples:")
print(f"{'Field':>6} {'Wire':>5} {'Tag (dec)':>10} {'Tag (hex)':>10} {'Varint bytes':>14}")
print("-" * 50)
examples = [(1, 0), (1, 2), (2, 0), (2, 2), (15, 0), (16, 0), (100, 2), (536870911, 0)]
for fn, wt in examples:
    tag = make_tag(fn, wt)
    enc = encode_varint(tag)
    print(f"{fn:>6} {wt:>5} {tag:>10} {hex(tag):>10} {len(enc):>8} bytes")

## 8. Serialization Round-Trip

The Protobuf lifecycle for ONNX models follows this pipeline:

```
┌──────────────┐    SerializeToString()    ┌──────────────┐    write()    ┌───────────┐
│  ModelProto   │─────────────────────────▶│  bytes object │────────────▶│ .onnx file │
│  (Python obj) │                          │  (in memory)  │             │  (on disk) │
└──────────────┘                           └──────────────┘             └───────────┘
       ▲                                          ▲                           │
       │         ParseFromString()                │          read()           │
       └──────────────────────────────────────────┴───────────────────────────┘
```

Under the hood, `onnx.save_model()` calls `SerializeToString()` and writes the result; `onnx.load_model()` reads bytes and calls `ParseFromString()`. The parsing is zero-copy for `bytes` fields (like `raw_data` in `TensorProto`), which is why ONNX can load large models efficiently.

### The Serialized Message Layout

A serialized Protobuf message is a concatenation of field encodings:

$$\text{Message bytes} = \bigcup_{i=1}^{N} (\text{tag}_i \;\|\; \text{value}_i)$$

For length-delimited fields (strings, bytes, sub-messages), the value is preceded by a length varint:

$$\text{Field}_i = \text{tag}_i \;\|\; \text{length}_i \;\|\; \text{data}_i$$

The total serialized size of a message is:

$$S = \sum_{i=1}^{N} \bigl(|\text{tag}_i| + |\text{len}_i| + |\text{data}_i|\bigr)$$

where $|\cdot|$ denotes byte length, and $|\text{len}_i| = 0$ for non-length-delimited types.

In [ ]:
from onnx import helper, TensorProto, checker
import numpy as np

a = helper.make_tensor_value_info("A", TensorProto.FLOAT, ["batch", 4])
b = helper.make_tensor_value_info("B", TensorProto.FLOAT, ["batch", 4])
c = helper.make_tensor_value_info("C", TensorProto.FLOAT, ["batch", 4])

node = helper.make_node("Add", inputs=["A", "B"], outputs=["C"])
graph = helper.make_graph([node], "add_graph", inputs=[a, b], outputs=[c])
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
model.producer_name = "protobuf_tutorial"
model.doc_string = "A + B element-wise addition"

raw_bytes = model.SerializeToString()
print(f"ModelProto type:    {type(model).__name__}")
print(f"IR version:         {model.ir_version}")
print(f"Serialized size:    {len(raw_bytes)} bytes")
print(f"First 60 hex chars: {raw_bytes[:30].hex()}")

In [ ]:
print("Byte-level dump (first 80 bytes):")
print(f"{'Offset':<8s} {'Hex':<48s} {'ASCII'}")
print("-" * 70)
for offset in range(0, min(len(raw_bytes), 80), 16):
    chunk = raw_bytes[offset:offset+16]
    hex_str = ' '.join(f'{b:02x}' for b in chunk)
    ascii_str = ''.join(chr(b) if 32 <= b < 127 else '.' for b in chunk)
    print(f"{offset:06x}  {hex_str:<48s} {ascii_str}")

print(f"\nTotal serialized length: {len(raw_bytes)} bytes")
print(f"\nDecoding first tag:")
first_byte = raw_bytes[0]
fn, wt, wt_name = decode_tag(first_byte)
print(f"  Byte 0x{first_byte:02x} -> field_number={fn}, wire_type={wt} ({wt_name})")

In [ ]:
parsed = onnx.ModelProto()
parsed.ParseFromString(raw_bytes)

checker.check_model(parsed)
print("Round-trip successful!")
print(f"  Graph name:     {parsed.graph.name}")
print(f"  Producer:       {parsed.producer_name}")
print(f"  Doc string:     {parsed.doc_string}")
print(f"  Num nodes:      {len(parsed.graph.node)}")
print(f"  Op type:        {parsed.graph.node[0].op_type}")
print(f"  Inputs:         {[i.name for i in parsed.graph.input]}")
print(f"  Outputs:        {[o.name for o in parsed.graph.output]}")

assert parsed.SerializeToString() == raw_bytes, "Bytes differ!"
print("\nByte-for-byte identity confirmed.")

## 9. Size Comparison: Protobuf vs JSON vs XML

To quantify Protobuf's compactness advantage, let's build models of increasing complexity and compare the serialized sizes across formats.

### Theoretical Overhead Per Field

| Format | Per-field overhead | Numeric values | Repeated fields |
|--------|-------------------|----------------|----------------|
| **Protobuf** | 1–2 byte tag | Native binary | Packed (no per-element tag) |
| **JSON** | Full key string + `:` + delimiters | Text digits | `[v1, v2, ...]` |
| **XML** | `<tag>...</tag>` pair | Text digits | Repeated elements |

For a tensor with $N$ float32 values, the expected sizes are:

$$S_{\text{protobuf}} \approx 4N + O(1) \;\text{bytes}$$

$$S_{\text{JSON}} \approx 10N \;\text{bytes}$$

$$S_{\text{XML}} \approx 20N \;\text{bytes}$$

The Protobuf advantage grows with tensor size because the constant metadata overhead is amortized:

$$\text{Compactness ratio} = \frac{S_{\text{JSON}}}{S_{\text{protobuf}}} \xrightarrow{N \to \infty} \frac{10}{4} = 2.5$$

In [ ]:
import json
from onnx import numpy_helper

def build_model_with_weights(n_params: int) -> onnx.ModelProto:
    """Build a model containing a weight tensor with n_params floats."""
    X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [1, n_params])
    Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, [1, n_params])
    W = numpy_helper.from_array(
        np.random.randn(n_params).astype(np.float32), name="W"
    )
    node = helper.make_node("Add", ["X", "W"], ["Y"])
    graph = helper.make_graph([node], "weighted", [X], [Y], initializer=[W])
    return helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])

def model_to_json_size(m: onnx.ModelProto) -> int:
    data = {
        "ir_version": int(m.ir_version),
        "graph": {
            "name": m.graph.name,
            "nodes": [{"op": n.op_type, "in": list(n.input), "out": list(n.output)}
                      for n in m.graph.node],
            "initializers": [{"name": init.name,
                              "data": numpy_helper.to_array(init).tolist()}
                             for init in m.graph.initializer],
        }
    }
    return len(json.dumps(data).encode())

param_counts = [10, 100, 1000, 5000, 10000, 50000]
pb_sizes, json_sizes = [], []

print(f"{'Params':>8} | {'Protobuf':>12} | {'JSON':>12} | {'Ratio':>7} | {'Theory (4N)':>12}")
print("-" * 62)
for n in param_counts:
    m = build_model_with_weights(n)
    pb = len(m.SerializeToString())
    js = model_to_json_size(m)
    pb_sizes.append(pb)
    json_sizes.append(js)
    print(f"{n:>8,} | {pb:>10,} B | {js:>10,} B | {js/pb:>6.1f}x | {4*n:>10,} B")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
x_pos = np.arange(len(param_counts))
width = 0.35
ax.bar(x_pos - width/2, [s/1024 for s in pb_sizes], width, label='Protobuf', color='#2196F3')
ax.bar(x_pos + width/2, [s/1024 for s in json_sizes], width, label='JSON', color='#FF9800')
ax.set_xlabel('Number of Parameters')
ax.set_ylabel('Size (KB)')
ax.set_title('Serialized Model Size', fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels([f'{n:,}' for n in param_counts], rotation=30)
ax.legend()
ax.grid(axis='y', alpha=0.3)

ax = axes[1]
ratios = [j/p for j, p in zip(json_sizes, pb_sizes)]
ax.plot(param_counts, ratios, 'o-', color='#4CAF50', linewidth=2, markersize=8)
ax.set_xlabel('Number of Parameters')
ax.set_ylabel('JSON / Protobuf Size Ratio')
ax.set_title('Compactness Advantage of Protobuf', fontweight='bold')
ax.set_xscale('log')
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"As models grow, JSON/Protobuf ratio converges to ~{ratios[-1]:.1f}x")

## 10. Visualizing Protobuf Message Trees

ONNX's Protobuf schema defines a hierarchy of nested messages. Understanding this tree is essential for navigating model files programmatically. The top-level `ModelProto` contains a `GraphProto`, which contains `NodeProto`s, `TensorProto`s (initializers), and `ValueInfoProto`s (inputs/outputs).

```
ModelProto
├── ir_version: int64
├── opset_import: repeated OperatorSetIdProto
│   ├── domain: string
│   └── version: int64
├── producer_name: string
├── producer_version: string
├── domain: string
├── model_version: int64
├── doc_string: string
├── graph: GraphProto
│   ├── name: string
│   ├── node: repeated NodeProto
│   │   ├── op_type: string
│   │   ├── input: repeated string
│   │   ├── output: repeated string
│   │   ├── name: string
│   │   ├── domain: string
│   │   └── attribute: repeated AttributeProto
│   │       ├── name: string
│   │       ├── type: AttributeType (enum)
│   │       ├── f / i / s / t / g: scalar values
│   │       └── floats / ints / strings: list values
│   ├── initializer: repeated TensorProto
│   │   ├── name: string
│   │   ├── dims: repeated int64
│   │   ├── data_type: int32
│   │   └── raw_data / float_data / ...
│   ├── input: repeated ValueInfoProto
│   │   ├── name: string
│   │   └── type: TypeProto
│   │       └── tensor_type: Tensor
│   │           ├── elem_type: int32
│   │           └── shape: TensorShapeProto
│   │               └── dim: repeated Dimension
│   ├── output: repeated ValueInfoProto
│   └── value_info: repeated ValueInfoProto
├── metadata_props: repeated StringStringEntryProto
│   ├── key: string
│   └── value: string
├── functions: repeated FunctionProto
└── training_info: repeated TrainingInfoProto
```

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('ONNX Protobuf Message Hierarchy', fontsize=16, fontweight='bold', pad=20)

colors = {
    'model': '#1565C0', 'graph': '#2E7D32', 'node': '#E65100',
    'tensor': '#6A1B9A', 'value': '#00838F', 'attr': '#C62828',
    'opset': '#37474F'
}

boxes = [
    (7, 9.2, 'ModelProto', colors['model']),
    (3, 7.2, 'opset_import[]\nOperatorSetIdProto', colors['opset']),
    (7, 7.2, 'GraphProto', colors['graph']),
    (11.5, 7.2, 'metadata_props[]\nStringStringEntryProto', colors['opset']),
    (3, 5.0, 'input[] / output[]\nValueInfoProto', colors['value']),
    (7, 5.0, 'node[]\nNodeProto', colors['node']),
    (11, 5.0, 'initializer[]\nTensorProto', colors['tensor']),
    (5, 3.0, 'attribute[]\nAttributeProto', colors['attr']),
    (9, 3.0, 'TypeProto\n(shape info)', colors['value']),
    (7, 1.2, 'TensorProto\n(embedded constant)', colors['tensor']),
]

for x, y, label, color in boxes:
    ax.text(x, y, label, ha='center', va='center', fontsize=9, fontweight='bold',
            color='white',
            bbox=dict(boxstyle='round,pad=0.5', facecolor=color, edgecolor='white', linewidth=1.5))

arrows = [
    (7, 8.8, 3, 7.7), (7, 8.8, 7, 7.7), (7, 8.8, 11.5, 7.7),
    (7, 6.8, 3, 5.4), (7, 6.8, 7, 5.4), (7, 6.8, 11, 5.4),
    (7, 4.6, 5, 3.4), (3, 4.6, 9, 3.4),
    (5, 2.6, 7, 1.6), (9, 2.6, 7, 1.6),
]
for x1, y1, x2, y2 in arrows:
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#546E7A', lw=1.5))

plt.tight_layout()
plt.show()

## 11. Schema Evolution and Compatibility

One of Protobuf's most valuable properties is its support for **schema evolution** — the ability to modify the schema over time while maintaining compatibility between old and new code. This is critical for ONNX, which must evolve its IR across versions without breaking the ecosystem.

### Formal Compatibility Definitions

Let $S_v$ denote the schema at version $v$, and let $\text{Parse}(S_v, b)$ denote parsing bytes $b$ under schema $S_v$:

**Forward compatibility** (old reads new): $\forall b \in \text{Valid}(S_{v+1}),\; \text{Parse}(S_v, b)$ succeeds. Unknown fields are preserved but not interpreted.

**Backward compatibility** (new reads old): $\forall b \in \text{Valid}(S_v),\; \text{Parse}(S_{v+1}, b)$ succeeds. Missing fields receive default values.

### Safe vs. Unsafe Schema Changes

```
┌────────────────────────────────────────┬───────────────────────────────────────┐
│           SAFE CHANGES                 │          UNSAFE CHANGES               │
├────────────────────────────────────────┼───────────────────────────────────────┤
│ Add new fields (new field numbers)     │ Change a field's type                 │
│ Add new enum values                    │ Change a field's number               │
│ Remove fields (reserve the number)     │ Reuse a deleted field number          │
│ Rename fields (wire uses numbers)      │ Change repeated ↔ scalar              │
│ Add new messages                       │ Remove a required field (proto2)      │
│ Change optional ↔ repeated (careful)   │ Change field's wire type              │
└────────────────────────────────────────┴───────────────────────────────────────┘
```

### ONNX's Three Versioning Axes

ONNX applies schema evolution at three independent levels:

1. **IR version** (`ModelProto.ir_version`): Controls structural changes to the proto schema itself. Bumped rarely, each bump is a breaking change for parsers that don't support the new IR.

2. **OpSet version** (`OperatorSetIdProto.version`): Controls operator semantics. A model can import multiple opsets (one per domain). Runtimes check whether they support the requested opset version.

3. **Model version** (`ModelProto.model_version`): User-defined, for tracking iterations of a specific model artifact. No semantic meaning to ONNX itself.

$$\text{Compatibility}(M) = \text{IR\_compat}(M.\text{ir\_version}) \wedge \bigwedge_{i} \text{OpSet\_compat}(M.\text{opset}_i)$$

In [ ]:
model_v1 = helper.make_model(
    helper.make_graph(
        [helper.make_node("Relu", ["X"], ["Y"])],
        "evolution_demo",
        [helper.make_tensor_value_info("X", TensorProto.FLOAT, [1, 10])],
        [helper.make_tensor_value_info("Y", TensorProto.FLOAT, [1, 10])],
    ),
    opset_imports=[helper.make_opsetid("", 13)]
)
v1_bytes = model_v1.SerializeToString()

model_v2 = onnx.ModelProto()
model_v2.ParseFromString(v1_bytes)
model_v2.producer_name = "tutorial_v2"
model_v2.producer_version = "2.0"
model_v2.doc_string = "Enhanced model with documentation"
entry = model_v2.metadata_props.add()
entry.key = "author"
entry.value = "ONNX Tutorial"

v2_bytes = model_v2.SerializeToString()

old_parser = onnx.ModelProto()
old_parser.ParseFromString(v2_bytes)
print("Forward compatibility demo:")
print(f"  v1 size: {len(v1_bytes)} bytes")
print(f"  v2 size: {len(v2_bytes)} bytes (added {len(v2_bytes)-len(v1_bytes)} bytes of metadata)")
print(f"  Graph still intact: {old_parser.graph.name}")
print(f"  Op still correct: {old_parser.graph.node[0].op_type}")
print(f"  Metadata preserved: {[(p.key, p.value) for p in old_parser.metadata_props]}")

## 12. Serialization Complexity Analysis

Understanding the computational complexity of Protobuf operations is important for performance-critical ML pipelines.

### Serialization: `SerializeToString()`

Serialization traverses the message tree once, encoding each field:

$$T_{\text{serialize}} = O\!\left(\sum_{\text{fields}} |\text{data}_i|\right) = O(S)$$

where $S$ is the total serialized size. For an ONNX model with $N$ weight parameters of size $s$ bytes each:

$$T_{\text{serialize}} = O(Ns + M)$$

where $M$ is the metadata size (graph structure, shapes, etc.), typically $M \ll Ns$.

### Deserialization: `ParseFromString()`

Parsing is also $O(S)$ — each byte is read exactly once. However, the constant factor differs:

- **Varint fields**: Decode loop with ~7 iterations max per value
- **Fixed-width fields**: Direct memory copy — $O(1)$ per field
- **Length-delimited fields**: Read length, then bulk copy payload
- **Sub-messages**: Recursive parse — adds stack depth but not extra passes

### Memory Complexity

The in-memory `ModelProto` object typically uses **more** memory than the serialized form due to Python object overhead and pointer-based data structures:

$$\text{RAM}_{\text{parsed}} \approx S + O(N_{\text{objects}} \cdot c_{\text{overhead}})$$

where $N_{\text{objects}}$ is the number of Protobuf sub-messages and $c_{\text{overhead}}$ is the per-object Python overhead (~100–200 bytes per object).

### External Data Threshold

Protobuf has a practical single-message size limit of approximately $2^{31} - 1$ bytes (~2 GB). ONNX's **external data** mechanism addresses this by storing large tensors in separate files, keeping the `.onnx` file (the Protobuf message) compact:

$$\text{If } S_{\text{weights}} > S_{\text{threshold}} \implies \text{use external\_data}$$

The default threshold is often set to $1{,}024$ bytes per tensor, but for models exceeding 2 GB total, external data is mandatory.

In [ ]:
import time

sizes = [1000, 10000, 100000, 500000, 1000000]
serialize_times = []
deserialize_times = []

print(f"{'Params':>10s} | {'Ser. (ms)':>10s} | {'Deser. (ms)':>12s} | {'Size (KB)':>10s} | {'MB/s (ser)':>10s}")
print("-" * 62)

for n in sizes:
    m = build_model_with_weights(n)

    t0 = time.perf_counter()
    for _ in range(5):
        raw = m.SerializeToString()
    ser_ms = (time.perf_counter() - t0) / 5 * 1000

    t0 = time.perf_counter()
    for _ in range(5):
        p = onnx.ModelProto()
        p.ParseFromString(raw)
    deser_ms = (time.perf_counter() - t0) / 5 * 1000

    size_kb = len(raw) / 1024
    throughput = (len(raw) / 1e6) / (ser_ms / 1000) if ser_ms > 0 else float('inf')
    print(f"{n:>10,} | {ser_ms:>9.2f} | {deser_ms:>11.2f} | {size_kb:>9.1f} | {throughput:>9.1f}")
    serialize_times.append(ser_ms)
    deserialize_times.append(deser_ms)

print("\nBoth serialize and deserialize show O(n) scaling with model size.")

## 13. Key Takeaways

1. **Protobuf is a binary, schema-first serialization format** where field names are replaced by small integer tags, producing compact payloads and enabling fast parsing — ideal for ML model interchange.

2. **The varint encoding** uses a continuation-bit scheme where each byte contributes 7 payload bits:

$$\text{value} = \sum_{i=0}^{n-1} (b_i \;\&\; \texttt{0x7F}) \cdot 128^i$$

   Small integers need fewer bytes, but negative values without ZigZag encoding are costly.

3. **Wire format tags** encode both field identity and value type in a single varint:

$$\text{tag} = (\text{field\_number} \ll 3) \;|\; \text{wire\_type}$$

   Field numbers 1–15 are cheapest (1-byte tags).

4. **ZigZag encoding** maps signed integers to unsigned via $(n \ll 1) \oplus (n \gg 31)$, ensuring small absolute values get compact varint encodings.

5. **Serialization and deserialization are both $O(n)$** in message size, with Protobuf achieving 20–100× faster parsing than JSON and 2–4× better compactness.

6. **ONNX's `.proto` file** defines the complete model structure — `ModelProto` → `GraphProto` → `NodeProto` / `TensorProto` / `ValueInfoProto` — and ships pre-compiled with the `onnx` Python package.

7. **Schema evolution** allows ONNX to add new fields without breaking existing parsers — forward and backward compatibility are built into the wire format.

8. **External data** is required when model weights exceed the ~2 GB Protobuf single-message limit, keeping the `.onnx` metadata file compact while storing weights in separate binary files.